<a href="https://colab.research.google.com/github/OSHEAG/Fundamentos-para-el-an-lisis-de-datos-/blob/main/Semana_1/01_carga_exploracion_caudal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 1 — Carga de Datos, Exploración Inicial y Visualización

## Proyecto: Análisis y Predicción del Caudal — Estación Pueblo Nuevo (DHIME)

**Objetivo:** Cargar los datos de caudal medio diario, explorar su estructura, limpiar columnas innecesarias, diagnosticar la completitud temporal y generar las primeras visualizaciones interactivas.

**Datos:** `descargaDhime.csv` — Estación PUEBLO NUEVO - AUT [21117100]  
**Variable:** Caudal medio diario (m³/s)  
**Período:** 2010 – 2017  

---

### Contenido
1. Importar librerías
2. Cargar y explorar datos crudos
3. Revisión de NivelAprobacion
4. Eliminar columnas innecesarias y preparar serie
5. Estadísticas descriptivas
6. Diagnóstico de completitud temporal (gaps)
7. Visualización: Serie de tiempo completa
8. Distribución del caudal (histograma + boxplot)
9. Caudal promedio por mes (estacionalidad)
10. Heatmap Año × Mes

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Importar Librerías

In [23]:

# =============================================================================
# CONCEPTO: Importación de librerías para análisis de series de tiempo
# -----------------------------------------------------------------------------
# Se importan librerías especializadas para el análisis de datos temporales
# e interactivos:
#   - pandas (pd):          Manipulación y análisis de datos; su DatetimeIndex
#                           es fundamental para trabajar con series de tiempo.
#   - numpy (np):           Operaciones numéricas y cálculo de estadísticos.
#   - plotly.express (px):  Gráficos interactivos de alto nivel con poca líneas
#                           de código (barras, histogramas, líneas, series).
#   - plotly.graph_objects (go): API de bajo nivel de Plotly; permite personalizar
#                           cada elemento del gráfico (trazas, ejes, anotaciones).
#   - make_subplots:        Crea cuadrículas de subgráficos en Plotly.
#   - warnings.filterwarnings('ignore'): Suprime advertencias no críticas para
#                           mantener la salida del notebook limpia.
#
# pio.templates.default = "plotly_white": establece el tema visual global de
# todos los gráficos Plotly como fondo blanco con rejilla tenue, adecuado para
# presentaciones y documentos técnicos.
#
# CRITERIO DE USO: Plotly es preferido sobre Matplotlib/Seaborn cuando los
# gráficos serán explorados interactivamente (zoom, hover, filtros) o publicados
# como HTML embebido en reportes.
# =============================================================================

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Tema visual global: aplica a todos los gráficos Plotly del notebook
import plotly.io as pio
pio.templates.default = "plotly_white"

print("✅ Librerías cargadas correctamente")
print(f"   Pandas: {pd.__version__}")
print(f"   NumPy: {np.__version__}")


✅ Librerías cargadas correctamente
   Pandas: 2.2.2
   NumPy: 2.0.2


## 2. Cargar y Explorar los Datos Crudos

El archivo `descargaDhime.csv` proviene del sistema DHIME del IDEAM (Colombia). Contiene datos de caudal medio diario medido en la estación hidrológica **Pueblo Nuevo**.

In [24]:

# =============================================================================
# CONCEPTO: Carga de datos crudos y exploración estructural inicial
# -----------------------------------------------------------------------------
# Se cargan siempre primero los datos en su estado crudo (sin procesar) para
# evaluar la calidad antes de tomar decisiones de limpieza.
#
# pd.read_csv(): carga el archivo tal como está, sin asumir tipos ni limpiar.
#   - Usar df_raw como nombre indica que el DataFrame contiene datos sin
#     procesar; convención útil para differencial el estado del dato.
#
# df.shape: tupla (filas, columnas) → verificación rápida de completitud.
# df.info(): muestra tipos, conteo de no-nulos y memoria → detecta columnas
#   con valores faltantes o tipos incorrectos (ej. fechas como 'object').
# df.head(10): visualiza los primeros 10 registros para entender la estructura
#   y verificar que el separador y codificación fueron interpretados correctamente.
#
# CRITERIO DE USO: Siempre cargar y revisar los datos crudos antes de
# transformarlos. Modificar los datos originales sin inspeccionarlos primero
# puede llevar a pérdida de información o errores difíciles de rastrear.
# =============================================================================

# Carga del archivo CSV tal como viene del sistema DHIME (IDEAM)
df_raw = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATOS/SEMANA1/descargaDhime.csv")

print(f"📋 Dimensiones: {df_raw.shape[0]} filas × {df_raw.shape[1]} columnas\n")
print("=" * 60)
print("INFORMACIÓN DEL DATASET")
print("=" * 60)
df_raw.info()   # Tipos de datos, conteo de no-nulos y uso de memoria
print("\n")
df_raw.head(10) # Primeros 10 registros para inspección visual


📋 Dimensiones: 2653 filas × 8 columnas

INFORMACIÓN DEL DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2653 entries, 0 to 2652
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CodigoEstacion   2653 non-null   int64  
 1   NombreEstacion   2653 non-null   object 
 2   Variable         2653 non-null   object 
 3   Parametro        2653 non-null   object 
 4   Fecha            2653 non-null   object 
 5   Unidad           2653 non-null   object 
 6   Valor            2653 non-null   float64
 7   NivelAprobacion  2653 non-null   object 
dtypes: float64(1), int64(1), object(6)
memory usage: 165.9+ KB




,CodigoEstacion,NombreEstacion,Variable,Parametro,Fecha,Unidad,Valor,NivelAprobacion
0,21117100,PUEBLO NUEVO - AUT [21117100],CAUDAL,Caudal medio diario,2010-01-01 00:00,m^3/s,1.196,Preliminar
1,21117100,PUEBLO NUEVO - AUT [21117100],CAUDAL,Caudal medio diario,2010-01-02 00:00,m^3/s,1.196,Preliminar
2,21117100,PUEBLO NUEVO - AUT [21117100],CAUDAL,Caudal medio diario,2010-01-03 00:00,m^3/s,1.196,Preliminar
3,21117100,PUEBLO NUEVO - AUT [21117100],CAUDAL,Caudal medio diario,2010-01-04 00:00,m^3/s,1.196,Preliminar
4,21117100,PUEBLO NUEVO - AUT [21117100],CAUDAL,Caudal medio diario,2010-01-05 00:00,m^3/s,1.196,Preliminar
5,21117100,PUEBLO NUEVO - AUT [21117100],CAUDAL,Caudal medio diario,2010-01-06 00:00,m^3/s,1.196,Preliminar
6,21117100,PUEBLO NUEVO - AUT [21117100],CAUDAL,Caudal medio diario,2010-01-07 00:00,m^3/s,1.196,Preliminar
7,21117100,PUEBLO NUEVO - AUT [21117100],CAUDAL,Caudal medio diario,2010-01-08 00:00,m^3/s,1.196,Preliminar
8,21117100,PUEBLO NUEVO - AUT [21117100],CAUDAL,Caudal medio diario,2010-01-09 00:00,m^3/s,1.196,Preliminar
9,21117100,PUEBLO NUEVO - AUT [21117100],CAUDAL,Caudal medio diario,2010-01-10 00:00,m^3/s,1.196,Preliminar


In [25]:

# =============================================================================
# CONCEPTO: Inspección del final del dataset y análisis de valores únicos
# -----------------------------------------------------------------------------
# Complementa la inspección inicial con información que head() no provee:
#
# df.tail(): muestra las últimas filas para verificar que:
#   - El archivo no tiene filas de totales, notas o metadatos al final.
#   - Los datos más recientes son coherentes con el período declarado.
#
# df.dtypes: confirma el tipo de dato de cada columna.
#   - 'object' en una columna de fechas indica que no fue parseada como datetime.
#   - 'object' en columnas numéricas indica posibles errores de formato (comas,
#     caracteres especiales) que impedirán cálculos numéricos directamente.
#
# df[col].nunique(): cuenta los valores distintos en cada columna.
#   - Una columna con nunique() == 1 tiene el mismo valor en todas las filas
#     → información redundante que puede eliminarse sin pérdida de datos.
#   - iloc[0]: muestra el primer valor como ejemplo representativo.
#
# CRITERIO DE USO: Sirve para identificar columnas candidatas a eliminación
# (valor único = sin variabilidad = sin información analítica) antes de limpiar.
# =============================================================================

print("🔽 Últimas 5 filas:")
display(df_raw.tail())   # Verifica que no hayan filas de resumen al final del CSV

print("\n📊 Tipos de datos:")
print(df_raw.dtypes)     # Detecta columnas mal tipadas (ej. fechas como 'object')

# Análisis de cardinalidad: columnas con 1 único valor son candidatas a eliminar
print("\n📊 Valores únicos por columna:")
for col in df_raw.columns:
    print(f"  {col}: {df_raw[col].nunique()} únicos → ejemplo: {df_raw[col].iloc[0]}")


🔽 Últimas 5 filas:


,CodigoEstacion,NombreEstacion,Variable,Parametro,Fecha,Unidad,Valor,NivelAprobacion
2648,21117100,PUEBLO NUEVO - AUT [21117100],CAUDAL,Caudal medio diario,2017-12-27 00:00,m^3/s,2.655625,Definitivo
2649,21117100,PUEBLO NUEVO - AUT [21117100],CAUDAL,Caudal medio diario,2017-12-28 00:00,m^3/s,2.287125,Definitivo
2650,21117100,PUEBLO NUEVO - AUT [21117100],CAUDAL,Caudal medio diario,2017-12-29 00:00,m^3/s,2.480417,Definitivo
2651,21117100,PUEBLO NUEVO - AUT [21117100],CAUDAL,Caudal medio diario,2017-12-30 00:00,m^3/s,2.655625,Definitivo
2652,21117100,PUEBLO NUEVO - AUT [21117100],CAUDAL,Caudal medio diario,2017-12-31 00:00,m^3/s,2.299167,Definitivo



📊 Tipos de datos:
CodigoEstacion       int64
NombreEstacion      object
Variable            object
Parametro           object
Fecha               object
Unidad              object
Valor              float64
NivelAprobacion     object
dtype: object

📊 Valores únicos por columna:
  CodigoEstacion: 1 únicos → ejemplo: 21117100
  NombreEstacion: 1 únicos → ejemplo: PUEBLO NUEVO - AUT [21117100]
  Variable: 1 únicos → ejemplo: CAUDAL
  Parametro: 1 únicos → ejemplo: Caudal medio diario
  Fecha: 2653 únicos → ejemplo: 2010-01-01 00:00
  Unidad: 1 únicos → ejemplo: m^3/s
  Valor: 1776 únicos → ejemplo: 1.196
  NivelAprobacion: 2 únicos → ejemplo: Preliminar


## 3. Revisión de la Columna `NivelAprobacion`

Antes de eliminar columnas, revisamos la proporción de datos **Preliminar** vs **Definitivo**.  
Esto indica la confiabilidad temporal: los datos más recientes suelen estar en estado *Preliminar*.

In [26]:

# =============================================================================
# CONCEPTO: Análisis de la columna de nivel de aprobación de datos
# -----------------------------------------------------------------------------
# Los datos hidrológicos del IDEAM tienen dos estados de aprobación:
#   - "Definitivo": datos revisados y validados por técnicos del IDEAM.
#   - "Preliminar": datos aún no validados (generalmente los más recientes).
#
# Es importante cuantificar esta proporción antes de eliminar la columna porque:
#   1. Indica qué fracción del dataset es confiable para análisis estadístico.
#   2. Permite decidir si excluir los datos preliminares del modelado final.
#   3. Los datos preliminares suelen concentrarse al final de la serie temporal.
#
# value_counts().reset_index(): cuenta la frecuencia de cada categoría y
#   convierte el resultado en un DataFrame para usarlo en Plotly.
#
# px.pie() con hole=0.3: gráfico de dona (donut chart). La versión con hueco
#   central es más legible que un pie chart tradicional cuando hay pocas
#   categorías, ya que el área central puede mostrar un total.
# textinfo="label+percent+value": muestra etiqueta, porcentaje y conteo
#   absoluto en cada sector del gráfico para máxima información.
#
# Rango de fechas por nivel: identifica en qué período temporal se
#   concentran los datos definitivos vs preliminares → patrón esperado es
#   que los datos más antiguos sean definitivos y los recientes preliminares.
#
# CRITERIO DE USO: Revisar el nivel de aprobación antes de cualquier análisis
# estadístico inferencial. Un alto porcentaje de datos preliminares puede
# introducir errores de medición que afecten modelos o hipótesis.
# =============================================================================

# Conteo de registros por nivel de aprobación
conteo_nivel = df_raw["NivelAprobacion"].value_counts().reset_index()
conteo_nivel.columns = ["NivelAprobacion", "Cantidad"]

# Gráfico de dona: más legible que pie chart para pocas categorías
fig = px.pie(
    conteo_nivel,
    names="NivelAprobacion",
    values="Cantidad",
    title="Proporción de Datos por Nivel de Aprobación",
    color_discrete_sequence=px.colors.qualitative.Set2,
    hole=0.3,   # Crea el espacio central del gráfico tipo dona
)
fig.update_traces(textinfo="label+percent+value")  # Muestra etiqueta + % + conteo
fig.update_layout(width=600, height=450)
fig.show()

# Rango temporal por nivel: confirma que los datos recientes son los preliminares
for nivel in df_raw["NivelAprobacion"].unique():
    subset = df_raw[df_raw["NivelAprobacion"] == nivel]
    print(f"  {nivel}: {subset['Fecha'].min()} → {subset['Fecha'].max()} ({len(subset)} registros)")


  Preliminar: 2010-01-01 00:00 → 2015-12-31 00:00 (1955 registros)
  Definitivo: 2016-01-01 00:00 → 2017-12-31 00:00 (698 registros)


## 4. Eliminación de Columnas Innecesarias y Preparación de la Serie

| Columna | Acción | Justificación |
|---------|--------|---------------|
| `CodigoEstacion` | ❌ Eliminar | Valor único (21117100) |
| `NombreEstacion` | ❌ Eliminar | Valor único (PUEBLO NUEVO) |
| `Variable` | ❌ Eliminar | Valor único (Caudal Medio Diario) |
| `Parametro` | ❌ Eliminar | Valor único (CDMD_D) |
| `Unidad` | ❌ Eliminar | Valor único (m³/s) |
| `NivelAprobacion` | ❌ Eliminar | Ya revisado, no aporta al análisis |
| `Fecha` | ✅ Mantener → **DatetimeIndex** | Eje temporal de la serie |
| `Valor` | ✅ Mantener → **Caudal** | Variable de estudio |

In [27]:

# =============================================================================
# CONCEPTO: Creación de la serie de tiempo y uso del DatetimeIndex
# -----------------------------------------------------------------------------
# Tras la exploración inicial se construye un DataFrame limpio con solo las
# columnas necesarias para el análisis de la serie de tiempo.
#
# Pasos y conceptos aplicados:
#
# 1. Selección y copia: df_raw[["Fecha", "Valor"]].copy()
#    - .copy() crea un DataFrame independiente para evitar el SettingWithCopyWarning
#      de pandas al modificar columnas del subconjunto.
#
# 2. Renombrado: .rename(columns={"Valor": "Caudal"})
#    - Nombre semántico: "Caudal" es más descriptivo que "Valor" genérico.
#
# 3. Conversión a datetime: pd.to_datetime()
#    - Transforma la columna de texto a tipo datetime64, habilitando:
#      operaciones temporales, resampleo, extracción de año/mes/día, etc.
#    - CRITERIO: Una columna de fecha como 'object' no permite ninguna
#      operación temporal; la conversión es obligatoria para series de tiempo.
#
# 4. set_index() + sort_index():
#    - Convierte la columna de fecha en el índice del DataFrame (DatetimeIndex).
#    - sort_index() garantiza que los registros estén en orden cronológico,
#      requisito para análisis de series de tiempo y detección de gaps.
#
# 5. inferred_freq: pandas intenta inferir automáticamente la frecuencia de
#    la serie (ej. 'D' = diaria). Si retorna None hay irregularidades o gaps.
#
# CRITERIO DE USO: Siempre establecer la columna de fecha como índice al
# trabajar con series de tiempo; habilita pd.date_range, resample(), shift(),
# rolling() y otros métodos específicos de series temporales en pandas.
# =============================================================================

# Construcción del DataFrame de serie de tiempo encadenando transformaciones
df = (
    df_raw[["Fecha", "Valor"]]
    .copy()                               # Copia independiente del subconjunto
    .rename(columns={"Valor": "Caudal"})  # Renombrado semántico
)

# Conversión de texto a datetime y establecimiento como índice temporal
df["Fecha"] = pd.to_datetime(df["Fecha"])   # 'object' → datetime64
df = df.set_index("Fecha").sort_index()     # DatetimeIndex ordenado cronológicamente

print(f"✅ Serie de tiempo creada:")
print(f"   Período: {df.index.min().date()} → {df.index.max().date()}")
print(f"   Registros: {len(df)}")
print(f"   Tipo de índice: {type(df.index).__name__}")
# inferred_freq: 'D' = diaria; None indica gaps que impiden inferir frecuencia
print(f"   Frecuencia detectada: {df.index.inferred_freq}")
print(f"\n📋 Primeras filas:")
df.head()


✅ Serie de tiempo creada:
   Período: 2010-01-01 → 2017-12-31
   Registros: 2653
   Tipo de índice: DatetimeIndex
   Frecuencia detectada: None

📋 Primeras filas:


,Caudal
Fecha,
2010-01-01,1.196
2010-01-02,1.196
2010-01-03,1.196
2010-01-04,1.196
2010-01-05,1.196


## 5. Estadísticas Descriptivas Básicas

Resumen numérico de la variable **Caudal (m³/s)** para entender la magnitud, dispersión y posibles valores extremos.

In [28]:

# =============================================================================
# CONCEPTO: Estadísticas descriptivas y métricas de dispersión para caudal
# -----------------------------------------------------------------------------
# df.describe() provee 8 estadísticos básicos para variables numéricas:
#   count, mean, std, min, 25%, 50% (mediana), 75%, max.
#
# Métricas adicionales calculadas:
#
#   Coeficiente de variación (CV) = std / mean × 100
#   - Mide la dispersión relativa en porcentaje de la media.
#   - CV > 100%: variabilidad extrema (típico en ríos con régimen torrencial).
#   - CV < 30%: baja variabilidad (régimen estable).
#   - Ventaja sobre std: permite comparar dispersión entre variables con
#     distintas unidades o escalas.
#
#   Rango = max - min
#   - Distancia entre el valor más alto y el más bajo.
#   - Sensible a outliers; si es mucho mayor que el IQR, hay valores extremos.
#
#   IQR (Rango Intercuartílico) = Q3 - Q1 (75% - 25%)
#   - Mide la dispersión del 50% central de los datos.
#   - Robusto a outliers (no depende de los valores extremos).
#   - Usado por el boxplot para definir el límite de outliers: 1.5 × IQR.
#
#   Valores NaN: df.isna().sum() cuenta los datos faltantes en la variable
#   objetivo; es crítico saberlo antes de calcular estadísticos o modelos
#   porque los NaN pueden sesgar o bloquear dichos cálculos.
#
# CRITERIO DE USO: Calcular CV e IQR para caudal porque la distribución de
# ríos es típicamente asimétrica (sesgo positivo); la media puede ser
# engañosa, y el IQR da una visión más robusta de la variabilidad central.
# =============================================================================

stats = df["Caudal"].describe()
print("📊 Estadísticas Descriptivas - Caudal (m³/s)")
print("=" * 45)
print(stats.to_string())   # .to_string() muestra todos los decimales sin truncar

# Métricas complementarias que describe() no incluye
print(f"\n📌 Coeficiente de variación: {(stats['std'] / stats['mean'] * 100):.1f}%")
print(f"📌 Rango (max - min): {stats['max'] - stats['min']:.2f} m³/s")
print(f"📌 IQR (Q3 - Q1): {stats['75%'] - stats['25%']:.2f} m³/s")
print(f"📌 Valores NaN en Caudal: {df['Caudal'].isna().sum()}")


📊 Estadísticas Descriptivas - Caudal (m³/s)
count    2653.000000
mean        3.402709
std         1.614810
min         0.081771
25%         2.652833
50%         3.330000
75%         3.983583
max        16.150000

📌 Coeficiente de variación: 47.5%
📌 Rango (max - min): 16.07 m³/s
📌 IQR (Q3 - Q1): 1.33 m³/s
📌 Valores NaN en Caudal: 0


## 6. Diagnóstico de Completitud Temporal

El dataset tiene **2,653 registros** pero el rango 2010–2017 debería tener ~2,922 días.  
Identificamos los **días faltantes** y cuantificamos los gaps por año para entender la calidad temporal.

In [29]:

# =============================================================================
# CONCEPTO: Diagnóstico de completitud temporal — detección de gaps
# -----------------------------------------------------------------------------
# Una serie de tiempo diaria debe tener exactamente un registro por día. Si
# faltan días (gaps), los cálculos de medias móviles, correlaciones y modelos
# de forecasting pueden producir resultados erróneos o fallar directamente.
#
# Estrategia:
#   1. pd.date_range(start, end, freq="D"): genera el rango COMPLETO de fechas
#      diarias esperadas entre la primera y última fecha de la serie.
#   2. rango_completo.difference(df.index): devuelve los días que están en el
#      rango esperado pero NO en el índice del DataFrame → días faltantes (gaps).
#   3. Completitud (%) = registros reales / días esperados × 100.
#      Umbral orientativo: se considera aceptable > 90–95% para series hidroló.
#
# Gaps por año: dt.year.value_counts().sort_index()
#   - Agrupa las fechas faltantes por año para identificar si los gaps se
#     concentran en un período específico (evento, falla de estación) o son
#     distribuidos a lo largo de toda la serie.
#
# Gráfico de barras con color_continuous_scale="Reds":
#   - Escala de color que refuerza visualmente la severidad: más oscuro = más
#     días faltantes. Facilita identificar los años más problemáticos.
#
# CRITERIO DE USO: Ejecutar siempre antes de cualquier resampleo, interpolación
# o modelado. Un gap no detectado puede generar autocorrelaciones artificiales
# o errores en ventanas de tiempo móviles.
# =============================================================================

# Paso 1: Generar el rango completo de días esperados
rango_completo = pd.date_range(start=df.index.min(), end=df.index.max(), freq="D")

# Paso 2: Comparar contra el índice real → días presentes en el rango pero no en los datos
fechas_faltantes = rango_completo.difference(df.index)

print(f"📅 Rango completo esperado: {len(rango_completo)} días")
print(f"📋 Días con datos: {len(df)}")
print(f"❌ Días faltantes: {len(fechas_faltantes)}")
print(f"📊 Completitud: {len(df) / len(rango_completo) * 100:.1f}%")

# Paso 3: Cuantificar gaps por año para identificar períodos críticos
gaps_year = pd.Series(fechas_faltantes).dt.year.value_counts().sort_index().reset_index()
gaps_year.columns = ["Año", "Días_Faltantes"]

print(f"\n📊 Días faltantes por año:")
print(gaps_year.to_string(index=False))

# Gráfico de barras: distribución temporal de los gaps
# color_continuous_scale="Reds": intensidad de color proporcional a la magnitud
fig = px.bar(
    gaps_year,
    x="Año",
    y="Días_Faltantes",
    title="Días Faltantes por Año",
    labels={"Días_Faltantes": "Días sin registro", "Año": "Año"},
    text="Días_Faltantes",          # Etiqueta del valor sobre cada barra
    color="Días_Faltantes",
    color_continuous_scale="Reds",  # Escala roja: más oscuro = más faltantes
)
fig.update_traces(textposition="outside")
fig.update_layout(width=800, height=450, showlegend=False)
fig.show()


📅 Rango completo esperado: 2922 días
📋 Días con datos: 2653
❌ Días faltantes: 269
📊 Completitud: 90.8%

📊 Días faltantes por año:
 Año  Días_Faltantes
2010              32
2012              62
2013               8
2014              65
2015              69
2016              33


In [30]:

# =============================================================================
# CONCEPTO: Identificación de gaps consecutivos (períodos sin datos)
# -----------------------------------------------------------------------------
# Saber QUÉ días faltan no es suficiente; es más útil saber si los días
# faltantes están aislados (un día aquí y allá) o agrupados en períodos
# prolongados sin medición.
#
# Un gap de 1 día puede imputarse fácilmente (interpolación lineal).
# Un gap de 30+ días requiere una estrategia diferente (medias históricas,
# modelos de imputación, o exclusión del período del análisis).
#
# Algoritmo de agrupación:
#   1. pd.DataFrame({"Fecha": fechas_faltantes}).sort_values("Fecha"):
#      convierte el índice de fechas faltantes en DataFrame y ordena.
#   2. .diff() > pd.Timedelta(days=1): detecta "saltos" de más de 1 día entre
#      fechas faltantes consecutivas. Un salto indica el inicio de un nuevo
#      grupo (gap distinto).
#   3. .cumsum(): acumula los saltos como un contador de grupo. Cada vez que
#      hay un salto, el contador sube → días continuos tienen el mismo grupo.
#
# groupby("Grupo").agg():
#   - Inicio: primera fecha del gap (min).
#   - Fin:    última fecha del gap (max).
#   - Días:   duración del gap en días (count).
#   sorted por Días descendente → los gaps más críticos aparecen primero.
#
# CRITERIO DE USO: Aplicar esta agrupación siempre que se identifiquen datos
# faltantes en una serie de tiempo. La duración del gap más largo determina
# qué método de imputación es viable en la Semana 2.
# =============================================================================

# Paso 1: Ordenar las fechas faltantes en un DataFrame
gaps_df = pd.DataFrame({"Fecha": fechas_faltantes}).sort_values("Fecha").reset_index(drop=True)

# Paso 2: Detectar límites de grupos — True cuando hay una brecha > 1 día entre fechas
# .diff() calcula la diferencia entre fechas consecutivas en el listado de faltantes.
# Si la diferencia es > 1 día, significa que el gap anterior terminó y empieza uno nuevo.
gaps_df["Grupo"] = (gaps_df["Fecha"].diff() > pd.Timedelta(days=1)).cumsum()

# Paso 3: Resumir cada grupo de gap con su inicio, fin y duración
gaps_resumen = gaps_df.groupby("Grupo").agg(
    Inicio=("Fecha", "min"),    # Primer día del gap
    Fin=("Fecha", "max"),       # Último día del gap
    Días=("Fecha", "count"),    # Duración en días
).sort_values("Días", ascending=False).reset_index(drop=True)   # Más largos primero

print("🔍 Gaps consecutivos más grandes (Top 10):")
print("=" * 55)
gaps_resumen.head(10)


🔍 Gaps consecutivos más grandes (Top 10):


,Inicio,Fin,Días
0,2014-11-15,2015-03-05,111
1,2012-11-01,2012-12-31,61
2,2016-08-14,2016-09-14,32
3,2010-01-12,2010-02-12,32
4,2013-12-25,2013-12-31,7
5,2014-08-01,2014-08-05,5
6,2014-08-08,2014-08-11,4
7,2014-07-24,2014-07-26,3
8,2015-06-26,2015-06-28,3
9,2014-02-28,2014-03-01,2


## 7. Serie de Tiempo Completa

Visualización de toda la serie de caudal diario (m³/s) para identificar **tendencias**, **estacionalidad** y **valores atípicos** a simple vista.

In [31]:

# =============================================================================
# CONCEPTO: Visualización de la serie de tiempo completa con Plotly
# -----------------------------------------------------------------------------
# El gráfico de línea de toda la serie es la primera visualización a generar
# en cualquier análisis de series de tiempo. Permite identificar a simple vista:
#   - Tendencia (¿el caudal sube o baja a largo plazo?).
#   - Estacionalidad (¿se repite un patrón cíclico cada año?).
#   - Anomalías o picos extremos (crecientes súbitas, errores de medición).
#   - Gaps visibles (quiebres abruptos en la línea).
#
# df.reset_index(): el índice DatetimeIndex debe convertirse en columna para
#   que Plotly pueda usarlo explícitamente en el eje x. Plotly Express trabaja
#   con columnas del DataFrame, no con el índice directamente.
#
# Parámetros clave de px.line():
#   line=dict(width=0.8): línea delgada para que las variaciones diarias sean
#     visibles sin saturar la gráfica cuando hay miles de puntos.
#
# rangeslider: agrega un control deslizante debajo del gráfico que permite
#   hacer zoom en períodos específicos sin perder el contexto global.
#   → Especialmente útil en series largas (años de datos diarios).
#
# hovermode="x unified": al pasar el cursor, muestra los valores de todas
#   las trazas en la misma posición del eje X en un único tooltip unificado,
#   facilitando la lectura precisa de valores.
#
# CRITERIO DE USO: Siempre como primer gráfico de una serie de tiempo.
# Plotly es preferido sobre Matplotlib aquí por el rangeslider interactivo.
# =============================================================================

# reset_index() convierte el DatetimeIndex en columna "Fecha" para Plotly
df_plot = df.reset_index()

fig = px.line(
    df_plot,
    x="Fecha",
    y="Caudal",
    title="Serie de Tiempo: Caudal Medio Diario – Estación Pueblo Nuevo (2010–2017)",
    labels={"Caudal": "Caudal (m³/s)", "Fecha": ""},
)
fig.update_traces(line=dict(width=0.8, color="#1f77b4"))  # Línea delgada y azul
fig.update_layout(
    width=1000,
    height=500,
    xaxis=dict(rangeslider=dict(visible=True)),  # Slider para zoom interactivo
    hovermode="x unified",                        # Tooltip unificado al hacer hover
)
fig.show()


## 8. Distribución del Caudal

Analizamos la forma de la distribución mediante un **histograma** y un **boxplot** para detectar asimetría y valores atípicos.

In [32]:

# =============================================================================
# CONCEPTO: Análisis de la distribución estadística del caudal
# -----------------------------------------------------------------------------
# Conocer la distribución del caudal es necesario antes de aplicar métodos
# estadísticos paramétricos (que asumen normalidad) o modelos de regresión.
#
# HISTOGRAMA con marginal="rug":
#   - Muestra la distribución de frecuencias del caudal.
#   - marginal="rug": agrega una franja de marcas en el eje X (rug plot) que
#     representa cada observación individualmente. Con muchos datos, permite
#     ver la densidad real sin depender solo de los bins.
#   - nbins=50: más bins que el defecto para capturar la forma con detalle.
#
# BOXPLOT con points="outliers":
#   - Resume la distribución en 5 estadísticos (min, Q1, mediana, Q3, max).
#   - points="outliers": muestra solo los puntos que superan 1.5 × IQR
#     (outliers), sin saturar el gráfico con todos los puntos.
#   - Complementa el histograma: mientras este muestra la forma, el boxplot
#     permite leer los cuartiles exactos y detectar outliers sistemáticamente.
#
# Asimetría (skewness):
#   - > 0: sesgo positivo (cola derecha larga → valores extremos altos).
#   - < 0: sesgo negativo (cola izquierda larga).
#   - |skewness| > 1: asimetría significativa → transformar antes de modelar.
#
# Curtosis (kurtosis):
#   - > 0: distribución leptocúrtica (cola más pesada que la normal, más outliers).
#   - < 0: distribución platicúrtica (colas más livianas que la normal).
#
# CRITERIO DE USO: Si skewness > 1, aplicar transformación log o Box-Cox
# antes de modelos que asuman normalidad (regresión lineal, ANOVA, ARIMA).
# =============================================================================

# Histograma con rug: visualiza la forma de la distribución y la densidad de puntos
fig_hist = px.histogram(
    df_plot,
    x="Caudal",
    nbins=50,
    title="Distribución del Caudal Medio Diario",
    labels={"Caudal": "Caudal (m³/s)", "count": "Frecuencia"},
    color_discrete_sequence=["#2ca02c"],
    marginal="rug",   # Franja de puntos individuales para ver densidad real
)
fig_hist.update_layout(width=900, height=450)
fig_hist.show()

# Boxplot: detecta cuartiles y outliers (puntos fuera de 1.5 × IQR)
fig_box = px.box(
    df_plot,
    y="Caudal",
    title="Boxplot del Caudal Medio Diario",
    labels={"Caudal": "Caudal (m³/s)"},
    color_discrete_sequence=["#ff7f0e"],
    points="outliers",  # Muestra solo outliers, no todos los puntos
)
fig_box.update_layout(width=500, height=450)
fig_box.show()

# Métricas de forma: informan si se requieren transformaciones previas al modelado
print(f"📐 Asimetría (skewness): {df['Caudal'].skew():.3f}")
print(f"📐 Curtosis (kurtosis): {df['Caudal'].kurtosis():.3f}")


📐 Asimetría (skewness): 1.401
📐 Curtosis (kurtosis): 6.901


## 9. Estacionalidad: Promedio Mensual de Caudal

El caudal en ríos colombianos suele mostrar un patrón **bimodal** (dos temporadas de lluvias).  
Calculamos el promedio mensual multianual para confirmar el patrón estacional.

In [33]:

# =============================================================================
# CONCEPTO: Estacionalidad — promedio mensual multianual (climatología)
# -----------------------------------------------------------------------------
# El promedio mensual multianual (también llamado "climatología del río") es
# el caudal medio de cada mes calculado sobre todos los años disponibles.
# Permite confirmar y cuantificar la estacionalidad del río.
#
# df.groupby(df.index.month): agrupa los registros por número de mes (1–12)
#   usando directamente el atributo .month del DatetimeIndex.
#   - .mean(): calcula la media del caudal para cada mes, promediando todos
#     los años del período (ej. todos los eneros 2010–2017).
#
# Lista de meses personalizados: se crea una lista "Mes" con abreviaciones
#   en español para usar como etiquetas en el eje X del gráfico, reemplazando
#   los números 1–12 que devuelve groupby().
#
# categoryorder="array" + categoryarray=meses: fuerza a Plotly a respetar el
#   orden cronológico de los meses (Ene → Dic), ya que por defecto px.bar
#   puede ordenar categorías alfabéticamente.
#
# color_continuous_scale="Blues": escala de color donde meses con mayor caudal
#   aparecen en azul oscuro, reforzando intuitivamente la idea de "más agua".
#
# CRITERIO DE USO: La climatología mensual es el primer paso para confirmar
# si la serie tiene estacionalidad antes de aplicar modelos SARIMA o
# descomposición STL. Si el patrón es bimodal (dos picos), se espera un
# ciclo anual de 12 meses con dos temporadas de lluvia.
# =============================================================================

# Abreviaciones de meses para etiquetas del eje X (orden cronológico español)
meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

# Promedio mensual multianual: media de cada mes sobre todos los años
monthly_avg = df.groupby(df.index.month)["Caudal"].mean().reset_index()
monthly_avg.columns = ["Mes_num", "Caudal_Promedio"]

# Mapeo de número de mes a etiqueta textual
monthly_avg["Mes"] = [meses[m - 1] for m in monthly_avg["Mes_num"]]

fig = px.bar(
    monthly_avg,
    x="Mes",
    y="Caudal_Promedio",
    title="Caudal Promedio Mensual Multianual (2010–2017)",
    labels={"Caudal_Promedio": "Caudal Promedio (m³/s)", "Mes": ""},
    text=monthly_avg["Caudal_Promedio"].round(1),   # Etiqueta de valor sobre cada barra
    color="Caudal_Promedio",
    color_continuous_scale="Blues",                  # Azul oscuro = caudal más alto
)
fig.update_traces(textposition="outside")
fig.update_layout(
    width=900,
    height=500,
    xaxis=dict(categoryorder="array", categoryarray=meses),  # Orden cronológico forzado
    showlegend=False,
)
fig.show()


In [34]:

# =============================================================================
# CONCEPTO: Boxplot mensual — variabilidad intra-mensual y comparación entre meses
# -----------------------------------------------------------------------------
# El promedio mensual multianual (gráfico anterior) muestra el valor central
# de cada mes, pero no comunica la variabilidad. Dos meses pueden tener el
# mismo promedio pero distribuciones muy distintas.
#
# El boxplot mensual muestra, para cada mes:
#   - Mediana: valor central del caudal en ese mes.
#   - Caja (Q1–Q3): rango donde se concentra el 50% de los datos.
#   - Bigotes: valores dentro de 1.5 × IQR (no outliers).
#   - Puntos: outliers del mes (valores extremos inusuales).
#
# Esto permite identificar:
#   - Meses con alta variabilidad interanual (caja grande) → difíciles de predecir.
#   - Meses con valores extremos (outliers) → picos de crecida específicos.
#   - Diferencias en la forma de la distribución entre temporada seca y lluviosa.
#
# Preparación:
#   df_monthly["Mes_num"] = df_monthly.index.month: extrae el número de mes del
#     DatetimeIndex directamente.
#   .map(dict(enumerate(meses, 1))): crea el diccionario {1:'Ene', 2:'Feb', ...}
#     y lo aplica para obtener la etiqueta de mes textual.
#
# color="Mes" + color_discrete_sequence: asigna un color único a cada mes para
#   distinguirlos visualmente; showlegend=False evita redundancia con el eje X.
#
# CRITERIO DE USO: Usar el boxplot mensual junto al promedio mensual para tener
# una visión completa: cuánto llueve en promedio (climatología) y qué tan
# variable es ese comportamiento año a año (variabilidad interanual).
# =============================================================================

df_monthly = df.copy()

# Extracción de mes usando el DatetimeIndex (no requiere columna de fecha separada)
df_monthly["Mes_num"] = df_monthly.index.month

# Mapeo número → etiqueta: dict(enumerate(meses, 1)) genera {1:'Ene', 2:'Feb', ...}
df_monthly["Mes"] = df_monthly["Mes_num"].map(dict(enumerate(meses, 1)))

fig = px.box(
    df_monthly.reset_index(),   # reset_index() expone Fecha para el hover
    x="Mes",
    y="Caudal",
    title="Distribución del Caudal por Mes (Boxplot)",
    labels={"Caudal": "Caudal (m³/s)", "Mes": ""},
    color="Mes",                                           # Color único por mes
    color_discrete_sequence=px.colors.qualitative.Set3,
    category_orders={"Mes": meses},                        # Orden cronológico forzado
)
fig.update_layout(width=1000, height=500, showlegend=False)
fig.show()


## 10. Heatmap: Caudal Promedio Año × Mes

Un **mapa de calor** permite visualizar simultáneamente la estacionalidad y la variación interanual.  
Las filas son los años y las columnas los meses.

In [35]:

# =============================================================================
# CONCEPTO: Heatmap Año × Mes — estacionalidad y variación interanual
# -----------------------------------------------------------------------------
# Un heatmap de caudal promedio cruzando año (filas) y mes (columnas) permite
# visualizar DOS dimensiones simultáneamente en una sola vista compacta:
#   - Lectura HORIZONTAL (por fila): variación del caudal a lo largo del año
#     → confirma la estacionalidad (patrón de meses húmedos y secos).
#   - Lectura VERTICAL (por columna): variación del mismo mes entre años
#     → detecta años anómalos (ej. un enero particularmente seco o lluvioso).
#
# Pipeline de preparación:
#   1. Agregar columnas "Año" y "Mes" al DataFrame usando el DatetimeIndex.
#   2. pivot_table(): reestructura el DataFrame para que el índice sean los
#      años, las columnas los meses y los valores el caudal promedio.
#      - aggfunc="mean": promedia los valores diarios de cada combinación año-mes.
#   3. pivot.columns = meses: renombra las columnas de números (1–12) a
#      abreviaciones legibles.
#
# px.imshow(): visualiza una matriz (DataFrame 2D) como heatmap de color.
#   - color_continuous_scale="YlGnBu": escala de amarillo (bajo) a azul oscuro
#     (alto), conveniente para representar volumen de agua.
#   - text_auto=".1f": muestra el valor promedio redondeado a 1 decimal en
#     cada celda del heatmap, combinando el mapa de color con datos exactos.
#   - aspect="auto": ajusta el tamaño de las celdas al espacio disponible.
#
# CRITERIO DE USO: Incluir este gráfico cuando se quiere comunicar de forma
# sintética tanto la estacionalidad como la variabilidad interanual en una
# sola imagen. Es especialmente útil en reportes y presentaciones.
# =============================================================================

heatmap_data = df.copy()
# Extracción de componentes temporales del DatetimeIndex
heatmap_data["Año"] = heatmap_data.index.year
heatmap_data["Mes"] = heatmap_data.index.month

# pivot_table: filas=Año, columnas=Mes, valores=promedio de Caudal
pivot = heatmap_data.pivot_table(values="Caudal", index="Año", columns="Mes", aggfunc="mean")
pivot.columns = meses  # Renombrar columnas numéricas a etiquetas textuales

fig = px.imshow(
    pivot,
    title="Heatmap: Caudal Promedio Mensual por Año",
    labels=dict(x="Mes", y="Año", color="Caudal (m³/s)"),
    color_continuous_scale="YlGnBu",  # Amarillo=bajo, azul oscuro=alto
    aspect="auto",                     # Ajusta celdas al espacio disponible
    text_auto=".1f",                   # Valor numérico en cada celda del mapa
)
fig.update_layout(width=900, height=450)
fig.show()


---

## Conclusiones de la Semana 1

### Hallazgos Clave:

1. **Datos crudos:** 2,653 registros de caudal medio diario (m³/s), estación PUEBLO NUEVO (IDEAM), período 2010–2017.
2. **Simplificación:** Se eliminaron 6 columnas con valores únicos/constantes. Solo se conservaron `Fecha` (índice) y `Valor` → `Caudal`.
3. **Completitud:** La serie tiene días faltantes (gaps) distribuidos en varios años. Esto se abordará con imputación en la **Semana 2**.
4. **Distribución:** El caudal presenta asimetría positiva (sesgo derecho), con presencia de valores extremos (crecientes).
5. **Estacionalidad:** Se observa un patrón estacional coherente con el régimen hidrológico colombiano.

### Próximos pasos (Semana 2):
- Reindexar a frecuencia diaria completa (`asfreq('D')`)
- Tratar valores faltantes con interpolación temporal y media móvil
- Aplicar transformaciones (log, Box-Cox) si es necesario
- Detectar y tratar outliers

In [36]:

# =============================================================================
# CONCEPTO: Validación numérica de completitud y extracción de datos del heatmap
# -----------------------------------------------------------------------------
# Esta celda genera un resumen tabular de la completitud por año y los valores
# exactos del heatmap para validar o exportar a reportes externos.
#
# Parte 1 — Completitud por año:
#   year % 4 == 0: detecta años bisiestos (366 días); años comunes tienen 365.
#   len(df[df.index.year == year]): filtra el DataFrame por año usando indexación
#     booleana sobre el DatetimeIndex (.year devuelve el año de cada fecha).
#   total_days - present: diferencia entre días esperados y presentes = faltantes.
#   → Permite identificar qué años tienen mayor porcentaje de datos faltantes
#     que deberán imputarse en la Semana 2.
#
# Parte 2 — Extracción de datos del heatmap:
#   pivot_table() recalcula la tabla Año × Mes de promedios.
#   pd.isna(v): verifica si un valor es NaN (combinación año-mes sin ningún dato)
#     y lo reemplaza con 0, evitando errores en serialización o exportación.
#   → Imprime los valores de cada fila del heatmap en formato de lista,
#     útil para validar que los promedios del gráfico son correctos o para
#     generar datos de prueba en análisis posteriores.
#
# CRITERIO DE USO: Usar esta celda como verificación final (sanity check)
# para confirmar que los cálculos del análisis exploratorio son consistentes
# antes de proceder a la limpieza e imputación de la Semana 2.
# =============================================================================

# --- Completitud por año ---
# Compara días presentes vs días esperados (considerando años bisiestos)
for year in range(2010, 2018):
    total_days = 366 if year % 4 == 0 else 365   # 2012 y 2016 son bisiestos
    present = len(df[df.index.year == year])      # Registros disponibles ese año
    print(f"{year}: present={present} missing={total_days - present}")

# --- Datos exactos del heatmap Año × Mes ---
# Reconstruye el pivot para extraer valores precisos de cada celda
hm = df.copy()
hm["Y"] = hm.index.year
hm["M"] = hm.index.month
pv = hm.pivot_table(values="Caudal", index="Y", columns="M", aggfunc="mean").round(3)

# Imprime cada fila como lista: útil para exportar o validar contra otros sistemas
for y in pv.index:
    vals = [round(v, 3) if not pd.isna(v) else 0 for v in pv.loc[y].tolist()]
    print(f"HM_{y}: {vals}")


2010: present=333 missing=32
2011: present=365 missing=0
2012: present=304 missing=62
2013: present=357 missing=8
2014: present=300 missing=65
2015: present=296 missing=69
2016: present=333 missing=33
2017: present=365 missing=0
HM_2010: [1.196, 2.47, 2.319, 3.494, 3.431, 3.301, 3.452, 3.209, 3.329, 3.657, 4.539, 8.492]
HM_2011: [3.63, 3.548, 4.956, 5.074, 4.988, 5.034, 4.869, 4.143, 3.787, 4.211, 3.721, 3.82]
HM_2012: [3.18, 3.002, 2.968, 3.263, 3.255, 3.28, 3.27, 3.268, 3.041, 3.167, 0, 0]
HM_2013: [3.687, 3.422, 3.721, 3.887, 4.066, 3.756, 4.133, 4.775, 4.758, 3.718, 4.229, 3.701]
HM_2014: [3.468, 2.846, 4.088, 3.696, 4.426, 4.585, 5.987, 7.199, 6.1, 5.538, 5.088, 0]
HM_2015: [0, 0, 2.358, 1.633, 1.565, 5.078, 3.979, 3.464, 2.772, 2.137, 2.294, 1.604]
HM_2016: [0.25, 0.155, 0.259, 0.727, 1.184, 1.54, 2.041, 1.683, 1.332, 1.452, 2.249, 1.841]
HM_2017: [4.056, 5.153, 3.826, 3.085, 3.075, 3.016, 3.552, 2.901, 2.765, 2.733, 2.666, 2.489]
